In [1]:
!pwd

/Users/scmps8/repos/github.com/ED_MRI/ED_MRI/examples


In [ ]:
#apply a previously trained model to a new dataset
from pathlib import Path


# This is the project directory where everything is saved - possible to get this automatically?)
#basedir = '/Users/paddyslator/python/ED_MRI/ED_MRI/examples/paper_experiments/ADC_model'

basedir = '/Users/scmps8/repos/github.com/ED_MRI/ED_MRI/results/paper_experiments/ADC_model'

trained_model_filename = "results/ADC_model_SNR20_n_train_vox_100_all_trained_task_network.pt" 

results_filename = "results/ADC_model_SNR20_n_train_vox_100_all.npy"

simulations_filenames = "data/ADC_model_SNR20_simulation_data.npy"

#basedir = "/home/blumberg/Bureau/z_Automated_Measurement/Output/tst/"



In [3]:
import torch
from omegaconf import OmegaConf
from tadred.trainer import Trainer
from tadred.data_processing import create_data_norm
from pathlib import Path


checkpoint = torch.load(
    Path(basedir,trained_model_filename),
    map_location="cpu",
    weights_only=False,
)

args = OmegaConf.create(checkpoint["args"])
data_features_norm = checkpoint["data_features_norm"]

nnet = Trainer(
    tadred_train_eval=args.tadred_train_eval,
    network=args.network,
    data_features_norm=data_features_norm,
    train_pytorch=args.train_pytorch,
    other_options=args.other_options,
)

nnet.device = "cpu"
nnet._create_model()
nnet.model.load_state_dict(checkpoint["model_state_dict"])
nnet.model.eval()




TADREDNet(
  (score_net): FCN(
    (layers): Sequential(
      (0): Linear(in_features=192, out_features=1000, bias=True)
      (1): ReLU()
      (2): Linear(in_features=1000, out_features=1000, bias=True)
      (3): ReLU()
      (4): Linear(in_features=1000, out_features=192, bias=True)
      (5): Identity()
    )
  )
  (task_net): FCN(
    (layers): Sequential(
      (0): Linear(in_features=192, out_features=1000, bias=True)
      (1): ReLU()
      (2): Linear(in_features=1000, out_features=1000, bias=True)
      (3): ReLU()
      (4): Linear(in_features=1000, out_features=1, bias=True)
      (5): Identity()
    )
  )
  (score_activation): Sigmoid()
  (downsampling_mult_layer): DownsamplingMultLayer()
  (loss_fct): MSELoss()
)

In [4]:
#load the datasets
import numpy as np
dataset = np.load(Path(basedir,simulations_filenames),allow_pickle=True).item()

FileNotFoundError: [Errno 2] No such file or directory: '/Users/scmps8/repos/github.com/ED_MRI/ED_MRI/results/paper_experiments/ADC_model/ADC_model_SNR20_simulation_data.npy'

In [ ]:
#apply the network to the dataset
test_output_nn = nnet.model.forward_eval(dataset['test'], score=1)

print(test_output_nn)


tensor([[0.6506],
        [2.5884],
        [1.9269],
        [0.2192],
        [2.0159],
        [0.8692],
        [1.9436],
        [1.9006],
        [1.1191],
        [1.1464]], grad_fn=<MulBackward0>)


In [ ]:
#load original tadred results
results = np.load(Path(basedir,results_filename),allow_pickle=True)

In [ ]:
results[12]['test_output']

array([[0.65057296],
       [2.5883613 ],
       [1.9268585 ],
       [0.2191986 ],
       [2.0158992 ],
       [0.8691753 ],
       [1.9435847 ],
       [1.900584  ],
       [1.119089  ],
       [1.1463583 ]], dtype=float32)

In [ ]:
#do the inference from the function directly
from tadred.inference import apply_trained_task_network 

test_output_nn = apply_trained_task_network(
    Path(basedir, trained_model_filename),
    Path(basedir, simulations_filenames),
)

print(test_output_nn)

FileNotFoundError: [Errno 2] No such file or directory: '/Users/scmps8/repos/github.com/ED_MRI/ED_MRI/examples/paper_experiments/ADC_model/results/ADC_model_SNR20_n_train_vox_100_all_trained_model.pt'